# 2 · Entrenamiento y evaluación

> **Tipo de ML:** `supervisado`  · Modelo activo: `RandomForest`


## 0. Entorno


In [22]:
# Verificar que el entorno está activo
import sys
print(f'Python: {sys.version}')
import xgboost; print(f'XGBoost: {xgboost.__version__}')



Python: 3.13.5 (main, May  5 2026, 21:05:52) [GCC 14.2.0]
XGBoost: 3.3.0


## 1. Imports


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from climasafeai.utils.paths import (
    PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, REPORTS_DIR,
)


from climasafeai.models.train_model import train_models, load_models
from climasafeai.models.predict_model import evaluate_models, predict_new, DECISION_THRESHOLD
from climasafeai.visualization.visualize import plot_feature_importance


## 2. Cargar datos procesados


In [24]:
# ── Carga de datos procesados ───────────────────────────────────────────────
# Si el pipeline aún no se ha ejecutado, corre primero: make data && make features
try:
    
    X_train = pd.read_csv(PROCESSED_DATA_DIR / 'X_train_calor.csv')
    X_test  = pd.read_csv(PROCESSED_DATA_DIR / 'X_test_calor.csv')
    y_train = pd.read_csv(PROCESSED_DATA_DIR / 'y_train_calor.csv').squeeze()
    y_test  = pd.read_csv(PROCESSED_DATA_DIR / 'y_test_calor.csv').squeeze()
    print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
    print(f'Clases: {sorted(y_train.unique().tolist())}')
    print(f'Balance (train): {y_train.value_counts(normalize=True).round(3).to_dict()}')
    
except FileNotFoundError as _e:
    raise FileNotFoundError(
        f"Datos no encontrados: {_e}\n"
        "Ejecuta primero: make data && make features\n"
        "O desde este notebook: !make data features"
    ) from _e


Train: (137916, 7)  |  Test: (34479, 7)
Clases: [0, 1, 2]
Balance (train): {0: 0.895, 1: 0.055, 2: 0.05}


## 3. Configuración


Modelo activo: `RandomForest`. Cambia `model_type` en json y regenera para entrenar otro.



In [25]:

# Configuración: model_type activo = 'RandomForest'
# tune_knn=True  → busca el k óptimo para KNN (lento en datasets grandes)
# cv_evaluate=True → muestra F1 5-fold de cada modelo antes de guardar
TUNE_KNN    = True
CV_EVALUATE = True


## 4. Entrenar


In [27]:
# ejecutar make mlflow-ui
models = train_models(X_train, y_train, tune_knn=TUNE_KNN, cv_evaluate=CV_EVALUATE)

--> Entrenando modelos de clasificacion...
    [RandomForest] entrenando...


2026/07/07 14:52:28 ERROR mlflow.store.db.utils: SQLAlchemy database error. The following exception is caught.
(raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO runs (run_uuid, name, source_type, source_name, entry_point_name, user_id, status, start_time, end_time, deleted_time, source_version, lifecycle_stage, artifact_uri, experiment_id) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('40971ceb60b442d9b578b22fc497c058', 'RandomForest', 'UNKNOWN', '', '', 'cacelas', 'RUNNING', 1783428746608, None, None, '', 'active', '/home/cacelas/Documentos/anfaia/ClimaSafeAI/notebooks/mlruns/1/40971ceb60b442d9b578b22fc497c058/artifacts', '1')]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Traceback (most recent call last):
  File "/home/cacelas/Documentos/anfaia/ClimaSafeAI/.venv/lib/python3.13/site-packag

MlflowException: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO runs (run_uuid, name, source_type, source_name, entry_point_name, user_id, status, start_time, end_time, deleted_time, source_version, lifecycle_stage, artifact_uri, experiment_id) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('40971ceb60b442d9b578b22fc497c058', 'RandomForest', 'UNKNOWN', '', '', 'cacelas', 'RUNNING', 1783428746608, None, None, '', 'active', '/home/cacelas/Documentos/anfaia/ClimaSafeAI/notebooks/mlruns/1/40971ceb60b442d9b578b22fc497c058/artifacts', '1')]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 5. Evaluar


In [ ]:

threshold  = DECISION_THRESHOLD   # ajustar si clases muy desbalanceadas
df_results = evaluate_models(
    models, X_train, y_train, X_test, y_test, threshold=threshold
)
df_results.sort_values('F1_test', ascending=False)


## 6. Importancia de variables


In [ ]:

feature_names = X_train.columns.tolist()
plot_feature_importance(models, feature_names)
from IPython.display import Image
Image(FIGURES_DIR / 'feature_importance.png')


## 7. Predicción sobre nuevos datos


In [ ]:

best_name  = df_results.sort_values('F1_test', ascending=False).iloc[0]['Modelo']
best_model = models[best_name]
print(f'Mejor modelo: {best_name}')

# Predecir sobre nuevos datos:
# from climasafeai.features.build_features import process_input
# X_new   = process_input(df_nuevos)
# preds, probs = predict_new(best_model, X_new, threshold=threshold)


## 8. Profiling (opcional)

Si el entrenamiento es lento, identifica dónde se va el tiempo:

```bash
make profile
snakeviz reports/profile.prof
```
